In [1]:
# !python3 run_preprocessing.py


In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [3]:
import os
BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'
print(os.listdir(BASE_DIR))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/data/초급_프로젝트/dataset'

In [ ]:
import json

with open(f'{BASE_DIR}/train_letterbox.json', 'r') as f:
    data = json.load(f)

print(list(data.keys()))
print(f"이미지 수: {len(data['images'])}")
print(f"어노테이션 수: {len(data['annotations'])}")
print(f"카테고리 수: {len(data['categories'])}")
print(data['images'][0])
print(data['annotations'][0])


In [ ]:
# !pip3 install transformers timm -q

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import json
import os

class DETRDataset(Dataset):
    def __init__(self, json_path, img_dir, transforms=None):
        with open(json_path, 'r') as f:
            coco = json.load(f)

        self.img_dir = img_dir
        self.transforms = transforms

        # 이미지 id → 파일명 매핑
        self.images = {img['id']: img for img in coco['images']}

        # 카테고리 id → 0부터 재매핑
        cats = sorted([c['id'] for c in coco['categories']])
        self.cat2idx = {c: i for i, c in enumerate(cats)}
        self.num_classes = len(cats)

        # 이미지별 어노테이션 묶기
        self.img_ids = list(self.images.keys())
        self.annots = {img_id: [] for img_id in self.img_ids}
        for ann in coco['annotations']:
            if ann['image_id'] in self.annots:
                self.annots[ann['image_id']].append(ann)

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]

        # 이미지 로드
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        image = Image.open(img_path).convert('RGB')
        W, H = image.size

        # 어노테이션 → DETR 포맷 (cx, cy, w, h 정규화)
        boxes, labels = [], []
        for ann in self.annots[img_id]:
            x, y, w, h = ann['bbox']
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            nw = w / W
            nh = h / H
            boxes.append([cx, cy, nw, nh])
            labels.append(self.cat2idx[ann['category_id']])

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.long),
            'image_id': torch.tensor([img_id])
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

In [ ]:
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images)
    return images, list(targets)

train_dataset = DETRDataset(
    json_path=f'{BASE_DIR}/train_letterbox.json',
    img_dir=f'{BASE_DIR}/letterbox_images/train',
    transforms=transform
)

val_dataset = DETRDataset(
    json_path=f'{BASE_DIR}/val_letterbox.json',
    img_dir=f'{BASE_DIR}/letterbox_images/val',
    transforms=transform
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

print(f"✅ train: {len(train_dataset)}장, val: {len(val_dataset)}장")
print(f"✅ 클래스 수: {train_dataset.num_classes}")

In [ ]:
from transformers import DetrForObjectDetection, DetrImageProcessor
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ device: {device}')

# pretrained DETR 로드 (COCO 학습된 모델)
model = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50',
    num_labels=73,
    ignore_mismatched_sizes=True
)
model.to(device)
print('✅ DETR 모델 로드 완료')

In [ ]:
from transformers import DetrForObjectDetection, DetrImageProcessor
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ device: {device}')

# pretrained DETR 로드 (COCO 학습된 모델)
model = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50',
    num_labels=73,
    ignore_mismatched_sizes=True
)
model.to(device)
print('✅ DETR 모델 로드 완료')

경고 내용 간단히 설명하면:

위 경고 → COCO pretrained 가중치 일부 불일치 (배치 정규화 관련) → 무시해도 됨
아래 경고 → 원래 COCO는 91클래스, 우리 건 73클래스라 분류기 레이어만 새로 초기화된 것 → 정상이고 의도한 동작

In [ ]:
from torch.optim import AdamW

optimizer = AdamW([
    {'params': model.model.parameters(), 'lr': 1e-5},   # backbone: 낮은 lr
    {'params': model.class_labels_classifier.parameters(), 'lr': 1e-4},  # 분류기: 높은 lr
], weight_decay=1e-4)

print('✅ Optimizer 준비 완료')

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for images, targets in loader:
        images = images.to(device)
        labels = [{'class_labels': t['labels'].to(device),
                   'boxes': t['boxes'].to(device)} for t in targets]

        outputs = model(pixel_values=images, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

print('✅ 학습 함수 준비 완료')

In [ ]:
EPOCHS = 30

for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}')

In [ ]:
!pip install torchmetrics -q

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate(model, loader, device):
    model.eval()
    metric = MeanAveragePrecision()

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)

            outputs = model(pixel_values=images)

            # DETR 출력 → torchmetrics 포맷 변환
            preds, gts = [], []
            for i, target in enumerate(targets):
                W, H = 800, 800
                # 예측
                scores = outputs.logits[i].softmax(-1)[:, :-1].max(-1)
                pred_labels = scores.indices
                pred_scores = scores.values
                # cx,cy,w,h → x1,y1,x2,y2
                boxes = outputs.pred_boxes[i]
                x1 = (boxes[:, 0] - boxes[:, 2] / 2) * W
                y1 = (boxes[:, 1] - boxes[:, 3] / 2) * H
                x2 = (boxes[:, 0] + boxes[:, 2] / 2) * W
                y2 = (boxes[:, 1] + boxes[:, 3] / 2) * H
                pred_boxes = torch.stack([x1, y1, x2, y2], dim=1)

                preds.append({'boxes': pred_boxes.cpu(),
                              'scores': pred_scores.cpu(),
                              'labels': pred_labels.cpu()})

                # GT
                gt_boxes = target['boxes']
                gt_x1 = (gt_boxes[:, 0] - gt_boxes[:, 2] / 2) * W
                gt_y1 = (gt_boxes[:, 1] - gt_boxes[:, 3] / 2) * H
                gt_x2 = (gt_boxes[:, 0] + gt_boxes[:, 2] / 2) * W
                gt_y2 = (gt_boxes[:, 1] + gt_boxes[:, 3] / 2) * H
                gt_boxes_xyxy = torch.stack([gt_x1, gt_y1, gt_x2, gt_y2], dim=1)

                gts.append({'boxes': gt_boxes_xyxy.cpu(),
                            'labels': target['labels'].cpu()})

            metric.update(preds, gts)

    result = metric.compute()
    print(f"✅ mAP@0.5     : {result['map_50']:.4f}")
    print(f"✅ mAP@0.5:0.95: {result['map']:.4f}")
    return result

evaluate(model, val_loader, device)

In [ ]:
import os
save_dir = '/content/drive/MyDrive/vera_detr'
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), f'{save_dir}/detr_epoch30.pth')
print('✅ 가중치 저장 완료!')

In [ ]:
test_dir = f'{BASE_DIR}/test_images'
print(os.listdir(test_dir)[:5])

In [ ]:
import pandas as pd
from PIL import Image
import torchvision.transforms as T
import torch
import os

# 역매핑: 0~72 → 원본 category_id
idx2cat = {v: k for k, v in train_dataset.cat2idx.items()}

transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

model.eval()
test_dir = f'{BASE_DIR}/test_images'
results = []
annotation_id = 1
SCORE_THRESHOLD = 0.5  # 이 이상 score만 제출

test_files = sorted(os.listdir(test_dir))
print(f'✅ 테스트 이미지 수: {len(test_files)}')

with torch.no_grad():
    for fname in test_files:
        image_id = int(os.path.splitext(fname)[0])
        img_path = os.path.join(test_dir, fname)

        image = Image.open(img_path).convert('RGB')
        W, H = image.size
        tensor = transform(image).unsqueeze(0).to(device)

        outputs = model(pixel_values=tensor)

        # 예측값 추출
        logits = outputs.logits[0]        # [100, 74]
        boxes  = outputs.pred_boxes[0]    # [100, 4] (cx,cy,w,h 정규화)

        scores_all = logits.softmax(-1)   # [100, 74]
        scores, labels = scores_all[:, :-1].max(-1)  # no-object 제외

        for score, label, box in zip(scores, labels, boxes):
            score = score.item()
            if score < SCORE_THRESHOLD:
                continue

            label = label.item()
            orig_cat_id = idx2cat.get(label, label)

            cx, cy, bw, bh = box.tolist()
            bbox_x = (cx - bw / 2) * W
            bbox_y = (cy - bh / 2) * H
            bbox_w = bw * W
            bbox_h = bh * H

            results.append({
                'annotation_id': annotation_id,
                'image_id':      image_id,
                'category_id':   orig_cat_id,
                'bbox_x':        round(bbox_x, 2),
                'bbox_y':        round(bbox_y, 2),
                'bbox_w':        round(bbox_w, 2),
                'bbox_h':        round(bbox_h, 2),
                'score':         round(score, 4),
            })
            annotation_id += 1

df = pd.DataFrame(results)
print(f'✅ 총 예측 수: {len(df)}')
print(df.head())

In [ ]:
import json
with open(f'{BASE_DIR}/train_letterbox.json') as f:
    data = json.load(f)

print(data['categories'][:5])

In [ ]:
save_path = '/content/drive/MyDrive/vera_detr/submission.csv'
df.to_csv(save_path, index=False)
print(f'✅ 저장 완료: {save_path}')

In [ ]:
# Early Stopping
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False

    def step(self, loss):
        if loss < self.best_loss:
            self.best_loss = loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

In [ ]:
# 실험 기록
import json, os

def save_experiment(config, train_losses, stopped_epoch, map_50, map_75):
    log = {
        'config': config,
        'stopped_epoch': stopped_epoch,
        'best_loss': min(train_losses),
        'map_50': round(map_50, 4),
        'map_75': round(map_75, 4),
        'train_losses': train_losses,
    }
    save_dir = '/content/drive/MyDrive/vera_detr/experiments'
    os.makedirs(save_dir, exist_ok=True)
    fname = f"lr{config['backbone_lr']}_cls{config['cls_lr']}.json"
    with open(os.path.join(save_dir, fname), 'w') as f:
        json.dump(log, f, indent=2)
    print(f'✅ 실험 기록 저장 완료: {fname}')

In [ ]:
def run_experiment(backbone_lr, cls_lr, epochs=30):
    from transformers import DetrForObjectDetection
    from torch.optim import AdamW

    config = {'backbone_lr': backbone_lr, 'cls_lr': cls_lr, 'epochs': epochs}
    exp_name = f"backbone{backbone_lr}_cls{cls_lr}"
    print(f"\n🚀 실험 시작: {config}")

    model = DetrForObjectDetection.from_pretrained(
        'facebook/detr-resnet-50',
        num_labels=73,
        ignore_mismatched_sizes=True
    ).to(device)

    optimizer = AdamW([
        {'params': model.model.parameters(), 'lr': backbone_lr},
        {'params': model.class_labels_classifier.parameters(), 'lr': cls_lr},
    ], weight_decay=1e-4)

    early_stopping = EarlyStopping(patience=10)
    train_losses = []

    for epoch in range(epochs):
        loss = train_one_epoch(model, train_loader, optimizer, device)
        train_losses.append(round(loss, 4))
        print(f'Epoch [{epoch+1}/{epochs}] Loss: {loss:.4f}')

        if early_stopping.step(loss):
            print(f'⏹️  Early Stopping @ epoch {epoch+1}')
            break

    # 가중치 + 학습 기록 저장
    save_dir = '/content/drive/MyDrive/vera_detr/experiments'
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), f'{save_dir}/{exp_name}.pth')

    import json
    with open(f'{save_dir}/{exp_name}_train_log.json', 'w') as f:
        json.dump({'config': config, 'train_losses': train_losses,
                   'stopped_epoch': epoch+1}, f, indent=2)

    print(f'✅ 저장 완료: {exp_name}.pth')
    return model

In [ ]:
model, result = run_experiment(backbone_lr=1e-4, cls_lr=1e-3)

In [ ]:
model = run_experiment(backbone_lr=5e-6, cls_lr=5e-5)

In [ ]:
result = evaluate(model, val_loader, device)